# The AG Share of New Company Registrations in Basel-Landschaft, 2017 vs 2025

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 2, Data and Statistical Reasoning

**Dataset:** "Firmenmutationen nach Rechtsform und Gemeinde" (company mutations by legal form and municipality), Open Government Data Kanton Basel-Landschaft:
https://data.bl.ch/explore/dataset/12460/

In this notebook I analyze new company registrations in the Swiss canton of
Basel-Landschaft.

**Research question:** Did the share of stock corporations (Aktiengesellschaft,
AG) among new company registrations in the canton of Basel-Landschaft change
between 2017 and 2025?

I picked this question because it connects to my first capstone project. In
Project 1 I analyzed the Swiss entries of the global LEI dataset and found
that the classic stock corporation (AG / SA) is the largest legal form group
in that register. This made me curious whether the AG also plays such a
strong role when new companies are founded, and whether that role changed
over time. One important point up front, the two projects use different
datasets and different populations. Project 1 looked at the stock of
LEI-registered entities, which is dominated by larger firms. This project
looks at single new registrations published in the Swiss Official Gazette of
Commerce (SHAB) for one canton. The percentages from the two projects must
not be compared directly.

## Dataset

The frozen snapshot `firmenmutationen.csv` is stored in the same folder as
this notebook. I downloaded it on 2026-08-13 through the direct CSV export:
https://data.bl.ch/explore/dataset/12460/download/?format=csv

The snapshot covers the period 2016-02-03 to 2026-08-12.

One row in this dataset is one single mutation message published in the
SHAB, so the data is not aggregated. The observation unit of my analysis is
one published new company registration. The online dataset is updated
continuously, so I froze a local copy and the notebook only reads this
local file. The publisher documents two known quality issues, the
municipality columns are partly incomplete and the free text field
`meldung` contains encoding errors. Both fields are not needed for my
research question.

## Setup

In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

## Load the dataset

I load the frozen CSV with a semicolon separator and check that the structure looks right before I touch anything.

In [48]:
df = pd.read_csv("firmenmutationen.csv", sep=";", encoding="utf-8")
df.head()

,kategorie,publikationsdatum_shab,journaldatum_handelsregister,id_shab,firmensitz_code,firmensitz,meldung,uid,firmenname,rechtsform_code,rechtsform
0,Neueintragung,2026-08-12,2026-08-07,1006729550,2762.0,Allschwil,"R.K. SanitÃ¤r Rima, in Allschwil, CHE-440.128....",CHE440128482,R.K. Sanitär Rima,101,Einzelunternehmen
1,Neueintragung,2026-08-12,2026-08-07,1006729549,2762.0,Allschwil,"R.K. Elektro Rima, in Allschwil, CHE-380.378.8...",CHE380378843,R.K. Elektro Rima,101,Einzelunternehmen
2,Neueintragung,2026-08-12,2026-08-07,1006729548,2834.0,Ziefen,"MS Swiss Handel GmbH, in Ziefen, CHE-285.436.8...",CHE285436803,MS Swiss Handel GmbH,107,Gesellschaft mit beschränkter Haftung (GmbH)
3,Löschung,2026-08-12,2026-08-07,1006729555,2824.0,Frenkendorf,"OlaVintage, Inhaber John Jairo Lopez, in Frenk...",CHE354942212,"OlaVintage, Inhaber John Jairo Lopez",101,Einzelunternehmen
4,Löschung,2026-08-12,2026-08-07,1006729554,2761.0,Aesch (BL),"MB Car Cleaning Jeyson Benitez Gomez, in Aesch...",CHE363228510,MB Car Cleaning Jeyson Benitez Gomez,101,Einzelunternehmen


The first rows look as expected, one register message per row with the mutation category, two dates, the company name, the legal form and a long free text field. The encoding problem the publisher mentions is directly visible, `meldung` shows broken characters like "SanitÃ¤r" while the separate `firmenname` column shows the correct "Sanitär". So the problem is only in the free text field, which I do not use.


In [49]:
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.info()

Rows: 28992, columns: 11
<class 'pandas.DataFrame'>
RangeIndex: 28992 entries, 0 to 28991
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kategorie                     28992 non-null  str    
 1   publikationsdatum_shab        28992 non-null  str    
 2   journaldatum_handelsregister  28992 non-null  str    
 3   id_shab                       28992 non-null  int64  
 4   firmensitz_code               28882 non-null  float64
 5   firmensitz                    28882 non-null  str    
 6   meldung                       28992 non-null  str    
 7   uid                           28992 non-null  str    
 8   firmenname                    28992 non-null  str    
 9   rechtsform_code               28992 non-null  int64  
 10  rechtsform                    28992 non-null  str    
dtypes: float64(1), int64(2), str(8)
memory usage: 2.4 MB


The file has 28,992 rows and 11 columns, which matches the download page. Three columns are numeric (`id_shab`, `firmensitz_code`, `rechtsform_code`), all of them are identifiers or codes. The two date columns are loaded as strings, so I will convert the publication date later. Nothing needs fixing at this point.

## Data checks

Before I filter anything I want to know how complete the data is, whether rows are duplicated and which values the two key columns `kategorie` and `rechtsform` actually contain.

In [50]:
df.isna().sum()

kategorie                         0
publikationsdatum_shab            0
journaldatum_handelsregister      0
id_shab                           0
firmensitz_code                 110
firmensitz                      110
meldung                           0
uid                               0
firmenname                        0
rechtsform_code                   0
rechtsform                        0
dtype: int64

Only the two municipality columns have missing values, 110 each. These are exactly the columns the publisher flags as partly incomplete. My analysis uses the mutation category, the legal form and the publication date, and all three are complete, so I do not drop or fill anything.

In [51]:
print("Complete duplicate rows:", df.duplicated().sum())
print("id_shab values are unique:", df["id_shab"].is_unique)

Complete duplicate rows: 0
id_shab values are unique: True


There are no complete duplicate rows and the SHAB message id is unique, so every row is one distinct register message.

In [52]:
df["kategorie"].value_counts()

kategorie
Neueintragung                13124
Löschung                      8356
Auflösung                     2704
Auflösung infolge Konkurs     1872
Widerruf der Auflösung        1767
Fusion                         566
Löschung infolge Fusion        549
Wiedereintragung                31
Spaltung                        23
Name: count, dtype: int64

The dataset covers nine mutation categories. My question is only about new company registrations, so the 13,124 rows with the category "Neueintragung" are the relevant subset. Deletions, mergers and the other categories describe events in the life of existing companies and are out of scope.

In [53]:
df["rechtsform"].value_counts()

rechtsform
Gesellschaft mit beschränkter Haftung (GmbH)           11272
Einzelunternehmen                                      10624
Aktiengesellschaft (AG)                                 5024
Kollektivgesellschaft (KlG)                              647
Zweigniederlassung                                       557
Stiftung                                                 336
Verein                                                   280
Genossenschaft                                           122
Zweigniederlassung eines ausländischen Unternehmens       85
Kommanditgesellschaft (KmG)                               40
Institut des öffentlichen Rechts                           3
Haupt von Gemeinderschaft                                  2
Name: count, dtype: int64

The legal form column has twelve clean values and no spelling variants. The exact label I need for the AG flag is "Aktiengesellschaft (AG)", and I take this string directly from the output above instead of guessing it.

## Cleaning

In this section I prepare the data in three small steps. I convert the
publication date from string to datetime, I reduce the data to the rows my
research question is about, and I add a flag column that marks the AG
registrations. Each step prints a small check, so nothing changes silently.

In [54]:
# convert the SHAB publication date from string to datetime and derive the year
df["publication_date"] = pd.to_datetime(df["publikationsdatum_shab"], errors="coerce")
df["year"] = df["publication_date"].dt.year

print("Dates that failed to convert:", df["publication_date"].isna().sum())
print("Date range:", df["publication_date"].min().date(), "to", df["publication_date"].max().date())

Dates that failed to convert: 0
Date range: 2016-02-03 to 2026-08-12


All 28,992 dates convert without a single failure, and the range confirms the note on the dataset page, the data starts in February 2016 and the current year is still running. Both border years are incomplete, so a fair yearly comparison must exclude them.

In [55]:
# keep only new company registrations, all other mutation types are out of scope
new_regs = df[df["kategorie"] == "Neueintragung"].copy()

print("New registrations in total:", len(new_regs))
print("UID values are unique within the new registrations:", new_regs["uid"].is_unique)

New registrations in total: 13124
UID values are unique within the new registrations: True


The UID is the official company identifier in Switzerland. Inside the 13,124 new registrations every UID appears exactly once, so no company shows up with a second registration in the covered period and I do not need any deduplication. One row really is one newly registered company.

In [56]:
# keep only the complete years, 2016 and 2026 are partial and would distort the comparison
new_regs = new_regs[new_regs["year"].between(2017, 2025)].copy()

print("New registrations 2017 to 2025:", len(new_regs))

New registrations 2017 to 2025: 11267


11,267 new registrations remain. To sum up the cleaning, I only excluded rows for two documented reasons. Rows with other mutation categories are outside the research question, and the years 2016 and 2026 are incomplete and would distort a yearly comparison. No rows were removed for quality reasons.

In [57]:
# flag the AG registrations, the basis for all share calculations below
new_regs["is_ag"] = new_regs["rechtsform"] == "Aktiengesellschaft (AG)"
print("AG registrations 2017 to 2025:", int(new_regs["is_ag"].sum()))

AG registrations 2017 to 2025: 1549


The flag `is_ag` marks the 1,549 AG registrations. With this the data preparation is done and I can start describing the data.